In [1]:
import threading
import pandas as pd
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
from datetime import datetime

class IBApp(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)
        self.data = []

    def historicalData(self, reqId, bar):
        self.data.append({
            "reqId": reqId,
            "date": bar.date,
            "open": bar.open,
            "high": bar.high,
            "low": bar.low,
            "close": bar.close,
            "volume": bar.volume
        })

    def historicalDataEnd(self, reqId, start, end):
        print(f"Finished reqId: {reqId}")

def run_loop(app):
    app.run()

app = IBApp()
app.connect("127.0.0.1", 7496, clientId=1)

# Start the IB message loop in a separate thread
thread = threading.Thread(target=run_loop, args=(app,), daemon=True)
thread.start()

def get_history(symbol, reqId=1):
    contract = Contract()
    contract.symbol = symbol
    contract.secType = "STK"
    contract.exchange = "SMART"
    contract.currency = "USD"

    app.data = []
    app.reqHistoricalData(
        reqId=reqId,
        contract=contract,
        endDateTime="",
        durationStr="8 Y",
        barSizeSetting="1 day",
        whatToShow="TRADES",
        useRTH=1,
        formatDate=1,
        keepUpToDate=False,
        chartOptions=[]
    )

    # Wait for data to return
    while len(app.data) == 0:
        pass

    return pd.DataFrame(app.data)

symbols = ['TLT',
            'VGLT',
            'VCLT',
            'SPLB',
            'MBB',
            'SPMB',
            'MUB',
            'VTEB']
dfs = []
for i, sym in enumerate(symbols):
    df = get_history(sym, reqId=i+1)
    df["symbol"] = sym
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)
full_df = full_df[["symbol", "date", "close"]]
full_df.head()

ERROR -1 2104 Market data farm connection is OK:uscrypto
ERROR -1 2104 Market data farm connection is OK:hfarm
ERROR -1 2104 Market data farm connection is OK:usfarm.nj
ERROR -1 2104 Market data farm connection is OK:jfarm
ERROR -1 2104 Market data farm connection is OK:usfuture
ERROR -1 2104 Market data farm connection is OK:cashfarm
ERROR -1 2104 Market data farm connection is OK:eufarmnj
ERROR -1 2104 Market data farm connection is OK:usfarm
ERROR -1 2106 HMDS data farm connection is OK:euhmds
ERROR -1 2106 HMDS data farm connection is OK:fundfarm
ERROR -1 2106 HMDS data farm connection is OK:ushmds
ERROR -1 2158 Sec-def data farm connection is OK:secdefnj


Finished reqId: 1
Finished reqId: 2
Finished reqId: 3
Finished reqId: 4
Finished reqId: 5
Finished reqId: 6
Finished reqId: 7
Finished reqId: 8


,symbol,date,close
0,TLT,20180228,118.75
1,TLT,20180301,119.32
2,TLT,20180302,118.35
3,TLT,20180305,118.03
4,TLT,20180306,118.14


In [2]:
df_pivot = full_df.pivot(index="date", columns="symbol", values="close")
df_pivot.head()

symbol,MBB,MUB,SPLB,SPMB,TLT,VCLT,VGLT,VTEB
date,,,,,,,,
20180227,NaN,NaN,NaN,25.47,NaN,NaN,NaN,NaN
20180228,104.42,108.61,27.02,25.50,118.75,90.80,73.08,50.69
20180301,104.38,108.70,26.90,25.49,119.32,90.30,73.46,50.76
20180302,104.08,108.38,26.79,25.49,118.35,89.80,72.83,50.67
20180305,103.95,108.41,26.80,25.46,118.03,89.79,72.66,50.61


In [3]:
df_pivot.to_csv("PRICES.csv")